# 🖼️ Qdrant Multimodal Search Demo

## What You'll Learn

This notebook demonstrates **text-to-image search** - using natural language queries to find relevant images.

## How Multimodal Search Works

```
Text Query: "beautiful mountain landscape"
                    ↓
            [Text Embeddings]
                    ↓
        Vector: [0.23, -0.45, ...]
                    ↓
    Search image captions/descriptions
                    ↓
    Return: Images of mountains! 🏔️
```

## What We're Demonstrating

| Feature | Description |
|---------|-------------|
| **Text → Image** | Type a description, find matching images |
| **Semantic matching** | "sunset" finds "golden hour" images |
| **Multilingual queries** | Search in Spanish, find any image |
| **Tag-based search** | Find images by concept tags |

## The Approach

Since text embedding models work with text, we store **image captions** (descriptions) as our searchable content. When you search for "mountain", we find image captions that mention mountains.

**Note:** For true image embedding (searching by image similarity), you'd use CLIP models. This demo uses text embeddings on image descriptions.

## Prerequisites

1. ✅ Ollama with `nomic-embed-text:latest`
2. ✅ Qdrant on port 6333
3. ✅ Llama Stack on port 8321 with Qdrant configured:
   ```bash
   OLLAMA_URL=http://localhost:11434/v1 QDRANT_URL=http://localhost:6333 llama stack run starter --port 8321
   ```

---
## Step 1: Connect to Llama Stack

**What:** Establish connection to the Llama Stack server.

**Expected:** Success message.

In [ ]:
import json
import io
from pathlib import Path
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8321/v1/", api_key="none")

print("✅ Connected to Llama Stack!")

---
## Step 2: Load Image Captions Data

**What we're doing:** Loading a JSON file containing image information:
- Filename
- English caption
- Spanish caption
- Italian caption
- Semantic tags

**Why:** These captions are what we'll embed and search. Each image has rich multilingual descriptions.

**Expected:** Display of all images with their captions.

In [ ]:
# Load image captions
captions_path = Path("../data/images/image_captions.json")
with open(captions_path, encoding="utf-8") as f:
    image_data = json.load(f)

print(f"📸 Loaded {len(image_data['images'])} images with captions:\n")

for img in image_data["images"]:
    print(f"🖼️  {img['id']}: {img['filename']}")
    print(f"   🇬🇧 EN: {img['caption']}")
    print(f"   🇪🇸 ES: {img['caption_es']}")
    print(f"   🏷️  Tags: {', '.join(img['tags'])}")
    print()

---
## Step 3: Display Sample Images (Optional)

**What we're doing:** If matplotlib/PIL is available, display the actual images.

**Why:** Visual context helps understand what we're searching for.

**Note:** This step is optional - the demo works without displaying images.

In [ ]:
try:
    from PIL import Image
    import matplotlib.pyplot as plt

    images_dir = Path("../data/images")
    num_images = len(image_data["images"])
    fig, axes = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
    if num_images == 1:
        axes = [axes]

    for idx, img_info in enumerate(image_data["images"]):
        img_path = images_dir / img_info["filename"]
        if img_path.exists():
            img = Image.open(img_path)
            axes[idx].imshow(img)
            axes[idx].set_title(f"{img_info['id']}\n{img_info['tags'][0]}", fontsize=10)
            axes[idx].axis("off")

    plt.tight_layout()
    plt.show()
    print("✅ Images displayed above!")

except ImportError:
    print("📝 Note: Install Pillow and matplotlib to display images:")
    print("   pip install Pillow matplotlib")
    print("\n   (Demo works fine without image display)")

---
## Step 4: Create Vector Store for Images

**What we're doing:** Creating a vector store that will hold our image captions.

**Why:** The captions will be embedded, allowing us to search for images using natural language.

**Expected:** Vector store ID.

In [ ]:
vector_store = client.vector_stores.create(
    name="multimodal_images",
    extra_body={
        "provider_id": "qdrant",
        "embedding_model": "ollama/nomic-embed-text:latest",
    },
)

print(f"✅ Created vector store: {vector_store.id}")
print("   This will store embeddings of image captions")

---
## Step 5: Insert Image Captions

**What we're doing:** For each image, we create a rich text document containing:
- Image ID and filename
- Captions in English, Spanish, and Italian
- Semantic tags

**Why:** By including multilingual captions, we enable searching in any language.

**Expected:** Confirmation of each image caption being inserted.

In [ ]:
print("📥 Inserting image captions into vector store...\n")

for img in image_data["images"]:
    # Create rich document with all caption information
    content = f"""Image ID: {img["id"]}
Filename: {img["filename"]}

Description (English): {img["caption"]}
Description (Spanish): {img["caption_es"]}
Description (Italian): {img["caption_it"]}

Tags: {", ".join(img["tags"])}
"""

    # Upload as file
    pseudo_file = io.BytesIO(content.encode("utf-8"))
    uploaded_file = client.files.create(
        file=(f"{img['id']}_caption.txt", pseudo_file, "text/plain"),
        purpose="assistants",
    )

    # Attach to vector store
    client.vector_stores.files.create(
        vector_store_id=vector_store.id, file_id=uploaded_file.id
    )

    print(f"   ✓ {img['id']}: {img['caption'][:50]}...")

print(f"\n✅ All {len(image_data['images'])} image captions inserted!")

---
## Step 6: Define Image Search Helper

**What we're doing:** Creating a helper function that searches for images and displays results clearly.

**Why:** Makes it easy to test different queries and see which images match.

In [ ]:
def search_images(query, query_language="English", max_results=3):
    """
    Search for images using a text query.

    Args:
        query: Natural language description of what you're looking for
        query_language: Language of the query (for display)
        max_results: Maximum number of images to return
    """
    print(f"\n{'=' * 70}")
    print(f"🔍 Query ({query_language}): '{query}'")
    print(f"{'=' * 70}")

    results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=query,
        max_num_results=max_results,
        extra_body={"search_mode": "vector"},
    )

    if not results.data:
        print("   ❌ No images found")
        return

    print(f"\n🖼️  Found {len(results.data)} matching images:\n")

    for i, result in enumerate(results.data, 1):
        content = result.content[0].text if result.content else ""

        # Extract image ID and description
        lines = content.split("\n")
        image_id = lines[0].replace("Image ID: ", "") if lines else "Unknown"

        # Find English description
        eng_desc = ""
        for line in lines:
            if line.startswith("Description (English):"):
                eng_desc = line.replace("Description (English): ", "")
                break

        print(f"   {i}. {image_id}")
        print(f"      Score: {result.score:.4f}")
        print(
            f"      {eng_desc[:80]}..." if len(eng_desc) > 80 else f"      {eng_desc}"
        )
        print()


print("✅ Search helper defined!")

---
## Step 7: Text-to-Image Search - English Queries

**What we're doing:** Searching for images using natural language descriptions in English.

**What to expect:**
- Semantic understanding: "mountain scenery" finds images described with similar concepts
- Results ranked by relevance (higher score = better match)

**Try different queries to see how the search understands your intent!**

In [ ]:
# Search for nature/landscape images
search_images("beautiful mountain scenery with snow")

print("💡 The search found images with mountain/nature descriptions,")
print("   even if they don't use the exact words 'scenery' or 'snow'.")

In [ ]:
# Search for urban/city images
search_images("city skyline at night with lights")

print("💡 This should find urban/architectural images.")

In [ ]:
# Search for beach/tropical images
search_images("tropical beach paradise vacation")

print("💡 Semantic search understands 'paradise' relates to beautiful beaches.")

---
## Step 8: Multilingual Image Search - Spanish Queries

**What we're doing:** Searching in Spanish to find the same images.

**Why this works:**
1. We stored Spanish captions with each image
2. The embedding model understands Spanish concepts
3. Spanish query vectors are close to Spanish caption vectors

**What to expect:** Same images found, regardless of query language!

In [ ]:
# Spanish: mountains with snow
search_images("montañas con nieve y cielo azul", query_language="Spanish")

print("💡 Spanish query found the same mountain images!")

In [ ]:
# Spanish: technology/digital
search_images("tecnología futurista digital", query_language="Spanish")

print("💡 Found tech-related images with Spanish query.")

---
## Step 9: Keyword Search for Tags & Exact Terms

**What we're doing:** Using keyword search to find images by exact tag names or specific words in captions.

**How it works:**
- Keyword search splits the query into words and matches any word literally
- Useful when you know exact tags ("nature", "city") or specific terms
- All matches get a fixed score of 1.0 (no similarity ranking)

**Comparison with vector search:**
- **Vector**: understands "winter scenery" relates to snow and mountains
- **Keyword**: only finds documents containing the literal word "winter" or "scenery"

In [ ]:
# Keyword search by tags
print("🏷️  KEYWORD SEARCH - Finding images by exact tags\n")

for tag_query in ["nature", "city urban", "technology"]:
    print(f"{'=' * 70}")
    print(f"🔍 Keyword query: '{tag_query}'")
    print(f"{'=' * 70}")

    results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=tag_query,
        max_num_results=3,
        extra_body={"search_mode": "keyword"},
    )

    if not results.data:
        print("   ❌ No results")
    else:
        for i, r in enumerate(results.data, 1):
            content = r.content[0].text if r.content else ""
            image_id = (
                content.split("\n")[0].replace("Image ID: ", "")
                if content
                else "Unknown"
            )
            print(f"   {i}. {image_id} | Score: {r.score:.4f}")
    print()

print("💡 Keyword search is great for finding images by known tags.")
print("   All matches score 1.0 since it's literal word matching.")

---
## Step 10: Conceptual Search & Vector vs Keyword Comparison

**What we're doing:** Comparing how vector and keyword search handle abstract concepts.

**Key insight:** Vector search excels at abstract/conceptual queries while keyword search needs exact words.

In [ ]:
# Compare vector vs keyword for a conceptual query
concept = "peaceful relaxing outdoor scenery"

print(f"📝 Query: '{concept}'\n")

for mode in ["vector", "keyword"]:
    print(f"{'=' * 70}")
    print(f"🔍 {mode.upper()} SEARCH")
    print(f"{'=' * 70}")

    results = client.vector_stores.search(
        vector_store_id=vector_store.id,
        query=concept,
        max_num_results=3,
        extra_body={"search_mode": mode},
    )

    if not results.data:
        print("   ❌ No results")
    else:
        for i, r in enumerate(results.data, 1):
            content = r.content[0].text if r.content else ""
            image_id = (
                content.split("\n")[0].replace("Image ID: ", "")
                if content
                else "Unknown"
            )
            eng_desc = ""
            for line in content.split("\n"):
                if line.startswith("Description (English):"):
                    eng_desc = line.replace("Description (English): ", "")[:60]
                    break
            print(f"   {i}. {image_id} | Score: {r.score:.4f} | {eng_desc}")
    print()

print("💡 Vector search understands 'peaceful relaxing' relates to nature/beach,")
print("   while keyword search needs exact word matches in the captions.")

---
## Step 11: Cleanup

**What we're doing:** Deleting the test vector store.

In [ ]:
client.vector_stores.delete(vector_store.id)
print(f"🗑️  Deleted vector store: {vector_store.id}")
print("\n✅ Multimodal demo complete!")

---
## 📚 Summary

### What We Demonstrated

| Feature | How It Works |
|---------|-------------|
| **Text to Image search** | Embed image captions, search with text queries |
| **Semantic matching** | "sunset" finds "golden hour" and "evening sky" images |
| **Multilingual queries** | Search in any language, find relevant images |
| **Keyword tag search** | Find images by exact tag names or specific words |
| **Vector vs Keyword** | Vector excels at conceptual queries; keyword needs exact words |

### Key Takeaways

1. **Rich captions matter**: The quality of your image descriptions directly affects search quality
2. **Multilingual captions**: Including captions in multiple languages enables global search
3. **Tags + keyword search**: Exact tag matching via keyword search complements semantic search
4. **Use the right mode**: Vector for conceptual queries, keyword for known tags, hybrid for both

### For True Multimodal (Image Embeddings)

This demo uses text embeddings on image captions. For true image-to-image search, use CLIP models:

```python
from fastembed import ImageEmbedding, TextEmbedding

# Embed images directly
image_model = ImageEmbedding(model_name="Qdrant/clip-ViT-B-32-vision")

# Embed text queries (same vector space as images)
text_model = TextEmbedding(model_name="Qdrant/clip-ViT-B-32-text")
```

### Next Steps

- Try `04_advanced_features_demo.ipynb` for filtering, thresholds, and chunking